# Feedforward Stochastic Discount Factor Network (PyTorch autodiff vs. manual gradients)

**UCLA MFE 413 — Machine Learning in Finance, coursework**
*(This assignment was completed as part of a study group; the analysis, PyTorch implementation, and the
auto-vs-manual gradient comparison below are my own submission.)*

### The problem
Under no-arbitrage, a stochastic discount factor (SDF) $m$ must price every excess return to zero:
$\mathbb{E}[m\, r^e_j] = 0$ for each asset $j$. The CPZ (Chen–Pelger–Zhu style) approach models the SDF as
$m_t = 1 - \sum_i \omega(x_{i,t};\theta_\omega)\, r^e_{t+1,i}$, where $\omega$ — the portfolio weight the
model assigns to each asset — is the output of a small neural network conditioned on firm characteristics
$x_{i,t}$. Training minimizes the sum of squared realized pricing errors
$J(\theta_\omega) = \tfrac12\sum_j s_j^2$, $s_j = \sum_t m_{t+1} r^e_{t+1,j}$.

### What this notebook does
The reference implementation (provided as a MATLAB script) hand-codes the full backward pass: nine
`gradient_omega` expressions, the chain rule through both sigmoid layers, and explicit
$\theta \leftarrow \theta - \eta\nabla_\theta L$ updates. Here it's reimplemented in PyTorch two ways, to
verify autodiff reproduces the hand-derived gradients exactly:

1. **`optimizer.step()`** — PyTorch's standard training loop.
2. **Manual update** — `loss.backward()` still does the autodiff, but the parameter update itself is
   written out by hand (`param -= eta * param.grad`), mirroring the MATLAB reference more directly.

Both use identical initialization, learning rate, and data, so the two loss curves should be
indistinguishable — which is itself the check that the manual differentiation in the original MATLAB code
and PyTorch's autodiff agree.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)

BigT = 10
BigN = 5
n2 = 2
Niter = 10000
eta = 2.0

### Data

Ten periods, five assets. Two per-asset-per-period characteristics — a market-return-like
variable (`x1`) and a second, larger-scale predictor (`x2`, e.g. an analyst-count-style
variable) — plus the realized excess returns each characteristic pair is meant to help price.
Both characteristics are normalized to be roughly $O(1)$ before entering the network.

In [ ]:
x1 = torch.tensor([
    [0.02, 0.02, 0.02, 0.02, 0.02],
    [0.0195, 0.0195, 0.0195, 0.0195, 0.0195],
    [0.019, 0.019, 0.019, 0.019, 0.019],
    [0.0185, 0.0185, 0.0185, 0.0185, 0.0185],
    [0.018, 0.018, 0.018, 0.018, 0.018],
    [0.0175, 0.0175, 0.0175, 0.0175, 0.0175],
    [0.017, 0.017, 0.017, 0.017, 0.017],
    [0.0165, 0.0165, 0.0165, 0.0165, 0.0165],
    [0.016, 0.016, 0.016, 0.016, 0.016],
    [0.0155, 0.0155, 0.0155, 0.0155, 0.0155]
], dtype=torch.float32)

x2 = torch.tensor([
    [72, 77, 88, 71, 44],
    [59, 66, 101, 62, 78],
    [65, 70, 79, 38, 33],
    [44, 47, 112, 25, 22],
    [40, 49, 88, 54, 12],
    [51, 55, 86, 50, 45],
    [44, 47, 55, 53, 24],
    [39, 44, 69, 43, 12],
    [68, 72, 65, 70, 74],
    [71, 75, 74, 70, 72]
], dtype=torch.float32)

Rexcess = torch.tensor([
    [0.03, 0.031, 0.032, 0.035, 0.04],
    [0.01, 0.0305, 0.0315, 0.0325, 0.0255],
    [-0.01, 0.03, 0.031, 0.0144, 0.07],
    [0.035, 0.0295, 0.0144, 0.0315, 0.06],
    [0.028, 0.029, 0.03, 0.035, 0.022],
    [0.015, 0.0285, 0.0295, 0.0305, 0.0315],
    [0.07, 0.028, 0.0144, 0.03, 0.2],
    [0.0265, 0.0275, 0.0285, 0.0295, 0.08],
    [0.026, 0.027, 0.028, 0.029, 0.05],
    [0.0144, 0.0265, 0.0144, 0.035, 0.049]
], dtype=torch.float32)

# Normalize
x1 = x1 / 0.02
x2 = x2 / 72

# Stack features so each observation has [x1, x2]
X = torch.stack([x1, x2], dim=2)   # BigT x BigN x 2

### Network and loss

A two-layer sigmoid network $\omega(x;\theta_\omega) = \sigma(W^{(3)}\sigma(W^{(2)}x+b^{(2)})+b^{(3)})$
mapping each asset's two characteristics to a portfolio weight in $(0,1)$. Initialized at the same
starting point as the MATLAB reference ($W^{(2)}=I$, $b^{(2)}=0$, $W^{(3)}=(1,1)$, $b^{(3)}=0$) so the two
implementations are directly comparable.

In [ ]:
class FeedforwardSDF(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Linear(2, 2)
        self.layer2 = torch.nn.Linear(2, 1)

        # MATLAB initialization:
        # W2 = [[1, 0], [0, 1]], b2 = [0, 0]
        # W3 = [1, 1], b3 = 0
        with torch.no_grad():
            self.layer1.weight.copy_(torch.tensor([[1.0, 0.0], [0.0, 1.0]]))
            self.layer1.bias.copy_(torch.tensor([0.0, 0.0]))
            self.layer2.weight.copy_(torch.tensor([[1.0, 1.0]]))
            self.layer2.bias.copy_(torch.tensor([0.0]))

    def forward(self, X):
        hidden = torch.sigmoid(self.layer1(X))
        omega = torch.sigmoid(self.layer2(hidden))
        return omega.squeeze(-1)


def sdf_loss(model, X, Rexcess):
    omega = model(X)  # BigT x BigN

    portfolio_return = torch.sum(omega * Rexcess, dim=1)  # sum across firms i
    m = 1 - portfolio_return                              # BigT

    s = torch.sum(m[:, None] * Rexcess, dim=0)             # one SDF moment per security j

    loss = 0.5 * torch.sum(s ** 2)
    return loss

### Training run 1 — `optimizer.step()`

In [ ]:
model_auto = FeedforwardSDF()
optimizer = torch.optim.SGD(model_auto.parameters(), lr=eta)

cost_auto = []

for counter in range(Niter):
    optimizer.zero_grad()
    loss = sdf_loss(model_auto, X, Rexcess)
    loss.backward()
    optimizer.step()

    cost_auto.append(loss.item())

plt.figure(figsize=(8, 5))
plt.semilogy(range(Niter), cost_auto)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Problem 5: PyTorch optimizer.step()")
plt.grid(True)
plt.show()

print("Final loss:", cost_auto[-1])

### Training run 2 — manual parameter update after `loss.backward()`

Same gradients (autodiff still computes them), but the update rule is written out explicitly rather than
delegated to `optimizer.step()` — the more direct PyTorch analogue of the MATLAB reference's manual
`theta <- theta - eta * grad` step.

In [ ]:
model_manual = FeedforwardSDF()

cost_manual = []

for counter in range(Niter):
    loss = sdf_loss(model_manual, X, Rexcess)

    # Clearing old gradients
    model_manual.zero_grad()

    # Computing gradients
    loss.backward()

    # Manual gradient descent update
    with torch.no_grad():
        for param in model_manual.parameters():
            param -= eta * param.grad

    cost_manual.append(loss.item())

plt.figure(figsize=(8, 5))
plt.semilogy(range(Niter), cost_manual)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Problem 5: Manual parameter update after loss.backward()")
plt.grid(True)
plt.show()

print("Final loss:", cost_manual[-1])

### Comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.semilogy(cost_auto, label="optimizer.step()")
plt.semilogy(cost_manual, label="manual update")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Problem 5: Comparison")
plt.legend()
plt.grid(True)
plt.show()

print("Final loss with optimizer.step():", cost_auto[-1])
print("Final loss with manual update:", cost_manual[-1])

### Takeaway

The network takes two normalized inputs, and outputs a portfolio weight $\omega(x_{t,i};\theta)$ for each
asset. The SDF moment condition and the sum-of-squared-pricing-errors loss are the same in both runs. In
the first version, PyTorch applies the parameter update through `optimizer.step()`; in the second,
PyTorch computes the gradients via `loss.backward()`, but the parameter update itself is written by hand.
Since the update rule is identical either way, **the two loss curves overlap exactly** — confirming that
PyTorch's autodiff reproduces the same gradients as the manual backward-pass derivation used in the
MATLAB reference, with a fraction of the code.